In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device="cuda"

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [9]:
def generate(model, tokenizer, prompt:str,
             max_token=50, **generate_kwargs):
    inputs=tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs=model.generate(**inputs, max_new_tokens=max_token, 
                           pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
prompt="List some places I should visit in paris"
generate(mistral7b, mistral7b_tokenizer, prompt)

["<s> List some places I should visit in paris?\n\nI'm going to Paris in a few weeks and I'm looking for some places to visit. I'm not really into museums and stuff like that. I'm more into the nightlife and the shopping. I"]

In [10]:
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""
full_prompt=f"{bob_introduction},\nuser:{prompt}, bob:"
extended_text=generate(mistral7b, mistral7b_tokenizer, full_prompt, max_token=100)
answer=extended_text[len(full_prompt):]
print(answer)



Bob:

The Eiffel Tower is a must-see for any visitor to Paris. It's an iconic symbol of the city and offers stunning views of the city from its observation deck.

The Louvre is one of the world's most famous museums and is home to some of the most famous works of art in the world, including the Mona Lisa.

Notre Dame Cathedral is a stunning example of Gothic architecture


["<s> \nBob is an amazing chatbot. It knows everything and it's incredibly helpful.\n,\nuser:List some places I should visit in paris, bob:\n\nBob:\n\nThe Eiffel Tower is a must-see for any visitor to Paris. It's an iconic symbol of the city and offers stunning views of the city from its observation deck.\n\nThe Louvre is one of the world's most famous museums and is home to some of the most famous works of art in the world, including the Mona Lisa.\n\nNotre Dame Cathedral is a stunning example of Gothic architecture"]